# ЛР1 · 01. Модели для временных рядов: zero-shot и длина контекста

Шесть предобученных моделей (протокол — `PROTOCOL.md`, п. 3–4):

| Модель | Веса |
|---|---|
| Chronos-Bolt Base | `amazon/chronos-bolt-base` |
| Chronos-2 | `amazon/chronos-2` (+ ковариаты, + соседи по графу) |
| TimesFM 2.5 | `google/timesfm-2.5-200m-pytorch` |
| Moirai 2.0 Small | `Salesforce/moirai-2.0-R-small` |
| Lag-Llama | `time-series-foundation-models/Lag-Llama` |
| TTM r2 | `ibm-granite/granite-timeseries-ttm-r2` |

Для каждой модели: короткая проверка на 200 окнах val (контекст 336 ч), затем тест (600 окон)
с контекстом 168 / 336 / 672 ч. Ошибка одной модели не останавливает остальные: она записывается
в `results/errors.csv` и считается результатом («модель не удалось запустить в среде Kaggle»).

Версия 2: исправлены TTM (явный выбор модели с окном 512 / 1024 ч) и Lag-Llama (совместимость с gluonts ≥ 0.15).

**Kaggle:** Inputs — датасет `vkr-transit-lab` и вывод ноутбука `lab-00-setup`; Accelerator — **GPU T4 x2**;
Internet — **On**. Запуск: Save Version → Save & Run All.

## 0. Установка библиотек

In [ ]:
import subprocess, sys
def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print("pip install", *args, "->", "OK" if r.returncode == 0 else "ОШИБКА\n" + r.stderr[-1500:])

pip("chronos-forecasting")                       # Chronos-Bolt, Chronos-2
pip("timesfm[torch]")                            # TimesFM 2.5
pip("--no-deps", "granite-tsfm")                 # TTM (без зависимостей, чтобы не трогать torch)
pip("--no-deps", "uni2ts==2.0.0")                # Moirai 2.0 (его зависимости требуют старый torch)
pip("einops", "jaxtyping", "hydra-core", "lightning", "gluonts")  # то, что реально нужно uni2ts и Lag-Llama
r = subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/time-series-foundation-models/lag-llama", "/tmp/lag-llama"],
                   capture_output=True, text=True)          # во временную папку, чтобы не попасть в Output
print("lag-llama clone:", "OK" if r.returncode == 0 else r.stderr[-500:])

# Совместимость Lag-Llama с gluonts >= 0.15: модуль gluonts.torch.modules.loss удалён,
# возвращаем его исходный код из gluonts 0.14.4 (классы DistributionLoss, NegativeLogLikelihood и др.).
import os, gluonts
loss_path = os.path.join(os.path.dirname(gluonts.__file__), "torch", "modules", "loss.py")
if not os.path.exists(loss_path):
    open(loss_path, "w").write('''import torch

from gluonts.pydantic import BaseModel


class DistributionLoss(BaseModel):
    """
    A ``torch.nn.Module`` extensions that computes loss values by comparing a
    ``Distribution`` (prediction) to a ``Tensor`` (ground-truth).
    """

    def __call__(
        self, input: torch.distributions.Distribution, target: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute the loss of predicting ``target`` with the ``input``
        distribution.

        Parameters
        ----------
        input
            Distribution object representing the prediction.
        target
            Tensor containing the ground truth.

        Returns
        -------
        torch.Tensor
            Tensor containing loss values, with the same shape as ``target``.

        Raises
        ------
        NotImplementedError
            [description]
        """
        raise NotImplementedError


class NegativeLogLikelihood(DistributionLoss):
    """
    Compute the negative log likelihood loss.

    Parameters
    ----------
    beta: float in range (0, 1)
        beta parameter from the paper: "On the Pitfalls of Heteroscedastic
        Uncertainty Estimation with Probabilistic Neural Networks" by
        Seitzer et al. 2022
        https://openreview.net/forum?id=aPOpXlnV1T
    """

    beta: float = 0.0

    def __call__(
        self, input: torch.distributions.Distribution, target: torch.Tensor
    ) -> torch.Tensor:
        nll = -input.log_prob(target)
        if self.beta > 0.0:
            variance = input.variance
            nll = nll * (variance.detach() ** self.beta)
        return nll


class CRPS(DistributionLoss):
    def __call__(
        self, input: torch.distributions.Distribution, target: torch.Tensor
    ) -> torch.Tensor:
        assert hasattr(input, "crps")
        return input.crps(target)


class QuantileLoss(DistributionLoss):
    def __call__(
        self, input: torch.distributions.Distribution, target: torch.Tensor
    ) -> torch.Tensor:
        assert hasattr(input, "quantile_loss")
        return input.quantile_loss(target)


class EnergyScore(DistributionLoss):
    def __call__(
        self, input: torch.distributions.Distribution, target: torch.Tensor
    ) -> torch.Tensor:
        assert hasattr(input, "energy_score")
        return input.energy_score(target)
''')
    print("gluonts.torch.modules.loss восстановлен:", loss_path)

In [ ]:
import glob, json, os, sys, time, traceback, gc, importlib.metadata
from pathlib import Path
import numpy as np, pandas as pd, torch

# labkit и подвыборка — из вывода ноутбука lab-00-setup
hits = glob.glob("/kaggle/input/**/labkit.py", recursive=True) or glob.glob("labkit.py")
LAB00 = Path(hits[0]).parent
sys.path.insert(0, str(LAB00))
import labkit as lk
print("labkit:", LAB00)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
for p in ["chronos-forecasting", "timesfm", "granite-tsfm", "uni2ts", "gluonts", "transformers"]:
    try:
        print(f"  {p}: {importlib.metadata.version(p)}")
    except Exception:
        print(f"  {p}: не установлен")

## 1. Данные, подвыборка, окна

In [ ]:
d = lk.load()
sub = json.load(open(LAB00 / "subset.json", encoding="utf-8"))
stations = sub["stations"]
test_orig = [pd.Timestamp(x) for x in sub["test_origins"]]
val_orig = [pd.Timestamp(x) for x in sub["val_origins"]]
assert stations == lk.make_subset(d), "подвыборка не совпадает с lab-00!"
tmax = d.train_max()

W = {("val", 336): lk.make_windows(d, stations, val_orig, context=336)}
for c in lk.CONTEXTS:
    W[("test", c)] = lk.make_windows(d, stations, test_orig, context=c)
for k, w in W.items():
    print(k, len(w), "окон, контекст", w.ctx.shape[1])
Q = list(lk.QUANTILES)

## 2. Общий запуск модели

In [ ]:
ERRORS = lk.results_dir() / "errors.csv"

def log_error(model, stage, exc):
    tb = traceback.format_exc()
    print(f"!!! {model} [{stage}] не отработал: {exc}\n{tb[-1500:]}")
    pd.DataFrame([{"date": pd.Timestamp.now(), "model": model, "stage": stage,
                   "error": repr(exc)[:500], "traceback": tb[-3000:]}]).to_csv(
        ERRORS, mode="a", header=not ERRORS.exists(), index=False)

def evaluate(model, mode, w, predict, batch=128, extra=None):
    """predict(ctx_batch) -> (point [b,24], quant [b,24,9] или None). Возвращает строку runs.csv."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t = lk.Timer()
    pts, qs = [], []
    for i in range(0, len(w), batch):
        with t:
            p, q = predict(w.ctx[i:i + batch], i)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        pts.append(np.asarray(p, dtype="float64"))
        qs.append(None if q is None else np.asarray(q, dtype="float64"))
    point = np.concatenate(pts)
    quant = None if qs[0] is None else np.concatenate(qs)
    # Постобработка для счётных данных: отрицательные значения обрезаются до 0 (PROTOCOL.md, п. 5)
    point = np.clip(point, 0, None)
    if quant is not None:
        quant = np.clip(np.sort(quant, axis=-1), 0, None)
    valid = lk.check_valid(list(point), w, tmax)
    point = lk.apply_fallback(list(point), valid, w)
    if quant is not None:
        quant[~valid] = point[~valid][..., None]
    timing = {}
    times = np.array(t.times) * 1000
    sizes = [min(batch, len(w) - i) for i in range(0, len(w), batch)]
    per = times / np.array(sizes)
    timing = {"ms_p50": float(np.median(per)), "ms_p95": float(np.quantile(per, 0.95)), "total_s": float(times.sum() / 1000)}
    row = lk.save_run(model, mode, w, point, valid, quant, timing,
                      {"context": w.ctx.shape[1], **(extra or {})})
    print(f"  {model:14s} {mode:22s} WAPE={row['wape']:.4f} [{row['wape_ci_lo']:.4f}; {row['wape_ci_hi']:.4f}] "
          f"невалидных={row['invalid_share']:.1%} p95={row['ms_p95']:.1f} мс/окно GPU={row['gpu_peak_gb']:.2f} ГБ")
    return row

def run_all(model, make_predict, extra=None):
    """make_predict(context_length) -> predict. Сначала val (контекст 336), затем тест на трёх контекстах."""
    plan = [("val", 336)] + [("test", c) for c in lk.CONTEXTS]
    for part, c in plan:
        try:
            pred = make_predict(c)
            evaluate(model, f"zs_ctx{c}_{part}", W[(part, c)], pred, extra=extra)
        except Exception as e:
            log_error(model, f"zs_ctx{c}_{part}", e)
            if part == "val":
                print(f"  {model}: на val не отработал — тест пропускается")
                return
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Chronos-Bolt Base

In [ ]:
try:
    from chronos import BaseChronosPipeline
    bolt = BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base", device_map=DEVICE, torch_dtype=torch.float32)

    def make_bolt(c):
        def predict(ctx, i):
            q, _ = bolt.predict_quantiles(torch.tensor(ctx, dtype=torch.float32), prediction_length=lk.HORIZON,
                                          quantile_levels=Q)
            q = q.cpu().numpy()
            return q[..., Q.index(0.5)], q
        return predict
    run_all("chronos_bolt", make_bolt, {"weights": "amazon/chronos-bolt-base"})
    del bolt
except Exception as e:
    log_error("chronos_bolt", "load", e)

## 4. Chronos-2

Три варианта на контексте 336 ч кроме основного zero-shot:
- **ковариаты**: календарь и погода (город) за прошлое и на 24 ч вперёд (погода — фактическая, допущение «идеального прогноза»);
- **соседи**: станция + 10 станций с наибольшим потоком поездок (`adj_od_top10`) как одна многомерная группа.

In [ ]:
COV_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_holiday", "is_school_day", "is_city_event",
            "temperature_2m", "precipitation", "snowfall"]
idx = d.y.index
cols = list(d.y.columns)
A = np.load(d.root / "graph" / "adj_od_top10.npy")
Yall = d.y.to_numpy(dtype="float64")
COV = d.cov[COV_COLS].to_numpy(dtype="float64")

def window_positions(w):
    return np.array([idx.get_loc(o) for o in w.meta["origin"]])

try:
    from chronos import Chronos2Pipeline
    c2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)

    def c2_out(inputs):
        q, _ = c2.predict_quantiles(inputs, prediction_length=lk.HORIZON, quantile_levels=Q, batch_size=128)
        q = np.stack([x[0].cpu().numpy() for x in q])          # первая (целевая) переменная: [b, 24, 9]
        return q[..., Q.index(0.5)], q

    def make_c2(c):
        def predict(ctx, i):
            return c2_out(ctx[:, None, :].astype("float32"))
        return predict
    run_all("chronos2", make_c2, {"weights": "amazon/chronos-2"})

    # --- ковариаты (контекст 336, тест) ---
    try:
        w = W[("test", 336)]
        pos = window_positions(w)
        def predict_cov(ctx, i):
            inputs = []
            for j in range(len(ctx)):
                p = pos[i + j]
                inputs.append({"target": ctx[j].astype("float32"),
                               "past_covariates": {n: COV[p - 336:p, k] for k, n in enumerate(COV_COLS)},
                               "future_covariates": {n: COV[p:p + lk.HORIZON, k] for k, n in enumerate(COV_COLS)}})
            return c2_out(inputs)
        evaluate("chronos2", "cov_ctx336_test", w, predict_cov, extra={"weights": "amazon/chronos-2",
                                                                         "covariates": ",".join(COV_COLS)})
    except Exception as e:
        log_error("chronos2", "cov_ctx336_test", e)

    # --- соседи по графу OD (контекст 336, тест) ---
    try:
        w = W[("test", 336)]
        pos = window_positions(w)
        nbrs = {s: [k for k in np.argsort(-A[cols.index(s)]) if A[cols.index(s), k] > 0][:10] for s in stations}
        def predict_nbr(ctx, i):
            inputs = []
            for j in range(len(ctx)):
                s, p = w.meta["station"].iat[i + j], pos[i + j]
                group = np.vstack([ctx[j], Yall[p - 336:p, nbrs[s]].T])     # [11, 336], строка 0 — сама станция
                inputs.append(group.astype("float32"))
            return c2_out(inputs)
        evaluate("chronos2", "graph_ctx336_test", w, predict_nbr, batch=32,
                 extra={"weights": "amazon/chronos-2", "graph": "adj_od_top10, 10 соседей"})
    except Exception as e:
        log_error("chronos2", "graph_ctx336_test", e)
    del c2
except Exception as e:
    log_error("chronos2", "load", e)

## 5. TimesFM 2.5

In [ ]:
try:
    import timesfm
    tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=False)
    tfm.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=256, normalize_inputs=True,
                                       use_continuous_quantile_head=True, fix_quantile_crossing=True,
                                       per_core_batch_size=64))
    def make_tfm(c):
        def predict(ctx, i):
            _, q = tfm.forecast(horizon=lk.HORIZON, inputs=[x.astype("float32") for x in ctx])
            q = np.asarray(q)[..., 1:10]          # индекс 0 — среднее, далее квантили 0,1…0,9
            return q[..., Q.index(0.5)], q
        return predict
    run_all("timesfm25", make_tfm, {"weights": "google/timesfm-2.5-200m-pytorch"})
    del tfm
except Exception as e:
    log_error("timesfm25", "load", e)

## 6. Moirai 2.0 Small

In [ ]:
try:
    from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module
    m2 = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")
    def make_moirai(c):
        model = Moirai2Forecast(module=m2, prediction_length=lk.HORIZON, context_length=c,
                                target_dim=1, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0).to(DEVICE)
        def predict(ctx, i):
            out = np.asarray(model.predict([x.astype("float32") for x in ctx]))   # [b, 9, 24] (+ лишняя ось)
            out = out.reshape(len(ctx), len(Q), lk.HORIZON)
            q = np.transpose(out, (0, 2, 1))
            return q[..., Q.index(0.5)], q
        return predict
    run_all("moirai2_small", make_moirai, {"weights": "Salesforce/moirai-2.0-R-small"})
    del m2
except Exception as e:
    log_error("moirai2_small", "load", e)

## 7. Lag-Llama

In [ ]:
try:
    os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"     # чекпойнт Lag-Llama хранит объекты gluonts
    sys.path.insert(0, "/tmp/lag-llama")
    from huggingface_hub import hf_hub_download
    from lag_llama.gluon.estimator import LagLlamaEstimator
    from gluonts.dataset.common import ListDataset
    ckpt_path = hf_hub_download("time-series-foundation-models/Lag-Llama", "lag-llama.ckpt")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    a = ckpt["hyper_parameters"]["model_kwargs"]

    def make_lagllama(c):
        est = LagLlamaEstimator(ckpt_path=ckpt_path, prediction_length=lk.HORIZON, context_length=c,
                                input_size=a["input_size"], n_layer=a["n_layer"], n_embd_per_head=a["n_embd_per_head"],
                                n_head=a["n_head"], scaling=a["scaling"], time_feat=a["time_feat"],
                                rope_scaling={"type": "linear", "factor": max(1.0, (c + lk.HORIZON) / a["context_length"])},
                                batch_size=64, num_parallel_samples=100, device=torch.device(DEVICE))
        predictor = est.create_predictor(est.create_transformation(), est.create_lightning_module())
        def predict(ctx, i):
            w = current_w[0]
            ds = ListDataset([{"start": pd.Period(w.meta["origin"].iat[i + j] - pd.Timedelta(hours=c), freq="h"),
                               "target": ctx[j].astype("float32")} for j in range(len(ctx))], freq="h")
            fc = list(predictor.predict(ds))
            q = np.stack([np.stack([f.quantile(t) for t in Q], axis=-1) for f in fc])    # [b, 24, 9]
            return q[..., Q.index(0.5)], q
        return predict

    current_w = [None]
    for part, c in [("val", 336)] + [("test", c) for c in lk.CONTEXTS]:
        try:
            current_w[0] = W[(part, c)]
            evaluate("lag_llama", f"zs_ctx{c}_{part}", W[(part, c)], make_lagllama(c),
                     extra={"weights": "time-series-foundation-models/Lag-Llama"})
        except Exception as e:
            log_error("lag_llama", f"zs_ctx{c}_{part}", e)
            if part == "val":
                break
except Exception as e:
    log_error("lag_llama", "load", e)
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 8. TTM r2 (точечный прогноз, без квантилей)

In [ ]:
try:
    from tsfm_public.toolkit.get_model import get_model
    def make_ttm(c):
        # явный выбор модели с окном 512 или 1024 ч (автоподбор в первой версии выбирал окно 180 ч)
        model = get_model("ibm-granite/granite-timeseries-ttm-r2", context_length=512 if c <= 512 else 1024,
                          prediction_length=lk.HORIZON).to(DEVICE).eval()
        L = model.config.context_length
        k = min(c, L)
        print(f"  TTM: контекст данных {c} ч, окно модели {L} ч")
        def predict(ctx, i):
            x = np.zeros((len(ctx), L), dtype="float32"); m = np.zeros_like(x)
            x[:, -k:] = ctx[:, -k:]; m[:, -k:] = 1                 # слева нули, помеченные как «не наблюдалось»
            with torch.no_grad():
                out = model(past_values=torch.tensor(x[..., None], device=DEVICE),
                            past_observed_mask=torch.tensor(m[..., None], device=DEVICE))
            return out.prediction_outputs[:, :lk.HORIZON, 0].float().cpu().numpy(), None
        return predict
    run_all("ttm_r2", make_ttm, {"weights": "ibm-granite/granite-timeseries-ttm-r2"})
except Exception as e:
    log_error("ttm_r2", "load", e)

## 9. Сводка

In [ ]:
runs = pd.read_csv(lk.results_dir() / "runs.csv")
cols_show = ["model", "mode", "n_windows", "invalid_share", "wape", "wape_ci_lo", "wape_ci_hi", "mase",
             "wql", "cov80", "ms_p95", "gpu_peak_gb"]
pd.set_option("display.width", 200)
print(runs[[c for c in cols_show if c in runs]].round(4).to_string())
if ERRORS.exists():
    print("\nОШИБКИ:")
    print(pd.read_csv(ERRORS)[["model", "stage", "error"]].to_string())
else:
    print("\nОшибок нет")